In [1]:
import pandas as pd
import sqlite3
import os

# Paths
BASE_DIR   = '/home/abdul/sales-analytics-dashboard'
CLEAN_DATA = os.path.join(BASE_DIR, 'data', 'processed', 'clean_superstore.csv')
DB_PATH    = os.path.join(BASE_DIR, 'data', 'processed', 'superstore.db')

# Load clean data
df = pd.read_csv(CLEAN_DATA)
print("✅ Clean data loaded")
print("Shape:", df.shape)

✅ Clean data loaded
Shape: (9994, 28)


In [2]:
#creating the sqlite database  and load the table
# Connect to SQLite (creates the .db file if it doesn't exist)
conn = sqlite3.connect(DB_PATH)

# Write dataframe as a SQL table called 'sales'
df.to_sql('sales', conn, if_exists='replace', index=False)

print("✅ Database created:", DB_PATH)

# Verify the table loaded correctly
verify = pd.read_sql("SELECT COUNT(*) AS total_rows FROM sales", conn)
print(verify)


✅ Database created: /home/abdul/sales-analytics-dashboard/data/processed/superstore.db
   total_rows
0        9994


In [3]:
#helper function to run queries cleanly
def run_query(sql, label=""):
    """Run a SQL query and return a styled dataframe."""
    result = pd.read_sql(sql, conn)
    if label:
        print(f"\n{'='*50}")
        print(f" {label}")
        print(f"{'='*50}")
    return result

In [5]:
#KPI Query 1: Overall business summary
q1 = run_query("""
    SELECT
        COUNT(DISTINCT "Order ID")    AS Total_Orders,
        COUNT(DISTINCT "Customer ID") AS Total_Customers,
        ROUND(SUM(Sales), 2)          AS Total_Revenue,
        ROUND(SUM(Profit), 2)         AS Total_Profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Overall_Margin_Pct,
        ROUND(AVG(Sales), 2)          AS Avg_Order_Value
    FROM sales
""", "Overall Business Summary")

q1


 Overall Business Summary


,Total_Orders,Total_Customers,Total_Revenue,Total_Profit,Overall_Margin_Pct,Avg_Order_Value
0,5009,793,2297200.86,286397.02,12.47,229.86


In [6]:
#KPI Query 2: Revenue and profit by year
q2 = run_query("""
    SELECT
        Year,
        COUNT(DISTINCT "Order ID")               AS Orders,
        ROUND(SUM(Sales), 2)                     AS Revenue,
        ROUND(SUM(Profit), 2)                    AS Profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Margin_Pct,
        ROUND(AVG(Sales), 2)                     AS Avg_Order_Value
    FROM sales
    GROUP BY Year
    ORDER BY Year
""", "Revenue & Profit by Year")

q2


 Revenue & Profit by Year


,Year,Orders,Revenue,Profit,Margin_Pct,Avg_Order_Value
0,2014,969,484247.50,49543.97,10.23,242.97
1,2015,1038,470532.51,61618.60,13.10,223.85
2,2016,1315,609205.60,81795.17,13.43,235.49
3,2017,1687,733215.26,93439.27,12.74,221.38


In [7]:
# KPI Query 3: Sales by region
q3 = run_query("""
    SELECT
        Region,
        COUNT(DISTINCT "Order ID")               AS Orders,
        ROUND(SUM(Sales), 2)                     AS Revenue,
        ROUND(SUM(Profit), 2)                    AS Profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Margin_Pct
    FROM sales
    GROUP BY Region
    ORDER BY Revenue DESC
""", "Performance by Region")

q3


 Performance by Region


,Region,Orders,Revenue,Profit,Margin_Pct
0,West,1611,725457.82,108418.45,14.94
1,East,1401,678781.24,91522.78,13.48
2,Central,1175,501239.89,39706.36,7.92
3,South,822,391721.91,46749.43,11.93


In [8]:
#KPI Query 4: Sales by category and sub-category
q4 = run_query("""
    SELECT
        Category,
        "Sub-Category",
        ROUND(SUM(Sales), 2)                     AS Revenue,
        ROUND(SUM(Profit), 2)                    AS Profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Margin_Pct,
        SUM(Quantity)                            AS Units_Sold
    FROM sales
    GROUP BY Category, "Sub-Category"
    ORDER BY Category, Profit DESC
""", "Revenue by Category & Sub-Category")

q4


 Revenue by Category & Sub-Category


,Category,Sub-Category,Revenue,Profit,Margin_Pct,Units_Sold
0,Furniture,Chairs,328449.10,26590.17,8.10,2356
1,Furniture,Furnishings,91705.16,13059.14,14.24,3563
2,Furniture,Bookcases,114880.00,-3472.56,-3.02,868
3,Furniture,Tables,206965.53,-17725.48,-8.56,1241
4,Office Supplies,Paper,78479.21,34053.57,43.39,5178
5,Office Supplies,Binders,203412.73,30221.76,14.86,5974
6,Office Supplies,Storage,223843.61,21278.83,9.51,3158
7,Office Supplies,Appliances,107532.16,18138.01,16.87,1729
8,Office Supplies,Envelopes,16476.40,6964.18,42.27,906
9,Office Supplies,Art,27118.79,6527.79,24.07,3000


In [9]:
#KPI Query 5: Top 10 customers by revenue
q5 = run_query("""
    SELECT
        "Customer Name",
        Segment,
        Region,
        COUNT(DISTINCT "Order ID")  AS Orders,
        ROUND(SUM(Sales), 2)        AS Total_Revenue,
        ROUND(SUM(Profit), 2)       AS Total_Profit
    FROM sales
    GROUP BY "Customer Name", Segment, Region
    ORDER BY Total_Revenue DESC
    LIMIT 10
""", "Top 10 Customers by Revenue")

q5


 Top 10 Customers by Revenue


,Customer Name,Segment,Region,Orders,Total_Revenue,Total_Profit
0,Sean Miller,Home Office,South,2,23669.20,-1787.04
1,Tamara Chand,Corporate,Central,2,18437.14,8745.06
2,Raymond Buch,Consumer,West,2,14345.28,6807.09
3,Tom Ashbrook,Home Office,East,2,13723.50,4599.21
4,Adrian Barton,Consumer,Central,5,12181.59,5362.61
5,Becky Martin,Consumer,Central,1,10539.90,-1878.79
6,Hunter Lopez,Consumer,East,2,10522.55,5045.86
7,Bill Shonely,Corporate,East,2,10022.29,2558.58
8,Sanjit Chand,Consumer,Central,1,9900.19,4668.69
9,Greg Tran,Consumer,East,5,9382.93,1749.11


In [10]:
#KPI Query 6: Impact of discount on profit
q6 = run_query("""
    SELECT
        CASE
            WHEN Discount = 0          THEN '0% - No discount'
            WHEN Discount <= 0.10      THEN '1-10%'
            WHEN Discount <= 0.20      THEN '11-20%'
            WHEN Discount <= 0.30      THEN '21-30%'
            ELSE 'Above 30%'
        END AS Discount_Band,
        COUNT(*)                               AS Orders,
        ROUND(AVG(Sales), 2)                   AS Avg_Sales,
        ROUND(AVG(Profit), 2)                  AS Avg_Profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Margin_Pct
    FROM sales
    GROUP BY Discount_Band
    ORDER BY Margin_Pct DESC
""", "Discount Impact on Profit")

q6


 Discount Impact on Profit


,Discount_Band,Orders,Avg_Sales,Avg_Profit,Margin_Pct
0,0% - No discount,4798,226.74,66.90,29.51
1,1-10%,94,578.40,96.06,16.61
2,11-20%,3709,213.58,24.74,11.58
3,21-30%,227,454.74,-45.68,-10.05
4,Above 30%,1166,222.59,-107.21,-48.16


In [11]:
#KPI Query 7: Monthly revenue trend
q7 = run_query("""
    SELECT
        YearMonth,
        Year,
        "Month Name",
        ROUND(SUM(Sales), 2)  AS Monthly_Revenue,
        ROUND(SUM(Profit), 2) AS Monthly_Profit
    FROM sales
    GROUP BY YearMonth, Year, "Month Name"
    ORDER BY YearMonth
""", "Monthly Revenue Trend")

q7


 Monthly Revenue Trend


,YearMonth,Year,Month Name,Monthly_Revenue,Monthly_Profit
0,2014-01,2014,Jan,14236.90,2450.19
1,2014-02,2014,Feb,4519.89,862.31
2,2014-03,2014,Mar,55691.01,498.73
3,2014-04,2014,Apr,28295.35,3488.84
4,2014-05,2014,May,23648.29,2738.71
5,2014-06,2014,Jun,34595.13,4976.52
6,2014-07,2014,Jul,33946.39,-841.48
7,2014-08,2014,Aug,27909.47,5318.10
8,2014-09,2014,Sep,81777.35,8328.10
9,2014-10,2014,Oct,31453.39,3448.26


In [12]:
#saving results to csv 
EXPORTS = os.path.join(BASE_DIR, 'exports')

q2.to_csv(os.path.join(EXPORTS, 'yearly_performance.csv'),    index=False)
q3.to_csv(os.path.join(EXPORTS, 'regional_performance.csv'),  index=False)
q4.to_csv(os.path.join(EXPORTS, 'category_performance.csv'),  index=False)
q5.to_csv(os.path.join(EXPORTS, 'top_customers.csv'),         index=False)
q6.to_csv(os.path.join(EXPORTS, 'discount_impact.csv'),       index=False)
q7.to_csv(os.path.join(EXPORTS, 'monthly_trend.csv'),         index=False)

print("✅ All query results saved to exports/")

✅ All query results saved to exports/


In [13]:
sql_script = """
-- ================================================
-- Sales Analytics Dashboard — KPI Queries
-- Dataset: Superstore Sales | Tool: SQLite
-- Author: Abdul
-- ================================================

-- 1. Overall business summary
SELECT
    COUNT(DISTINCT "Order ID")    AS Total_Orders,
    COUNT(DISTINCT "Customer ID") AS Total_Customers,
    ROUND(SUM(Sales), 2)          AS Total_Revenue,
    ROUND(SUM(Profit), 2)         AS Total_Profit,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Overall_Margin_Pct
FROM sales;

-- 2. Revenue and profit by year
SELECT Year, ROUND(SUM(Sales),2) AS Revenue,
    ROUND(SUM(Profit),2) AS Profit,
    ROUND(SUM(Profit)/SUM(Sales)*100,2) AS Margin_Pct
FROM sales GROUP BY Year ORDER BY Year;

-- 3. Performance by region
SELECT Region, ROUND(SUM(Sales),2) AS Revenue,
    ROUND(SUM(Profit),2) AS Profit
FROM sales GROUP BY Region ORDER BY Revenue DESC;

-- 4. Category and sub-category breakdown
SELECT Category, "Sub-Category",
    ROUND(SUM(Sales),2) AS Revenue,
    ROUND(SUM(Profit),2) AS Profit
FROM sales GROUP BY Category, "Sub-Category" ORDER BY Profit DESC;

-- 5. Discount impact on profit
SELECT
    CASE
        WHEN Discount = 0     THEN '0% - No discount'
        WHEN Discount <= 0.10 THEN '1-10%'
        WHEN Discount <= 0.20 THEN '11-20%'
        WHEN Discount <= 0.30 THEN '21-30%'
        ELSE 'Above 30%'
    END AS Discount_Band,
    ROUND(AVG(Profit),2) AS Avg_Profit,
    ROUND(SUM(Profit)/SUM(Sales)*100,2) AS Margin_Pct
FROM sales GROUP BY Discount_Band;
"""

sql_path = os.path.join(BASE_DIR, 'sql', 'kpi_queries.sql')
with open(sql_path, 'w') as f:
    f.write(sql_script)

print("✅ SQL file saved to sql/kpi_queries.sql")

# Close database connection
conn.close()
print("✅ Database connection closed")

✅ SQL file saved to sql/kpi_queries.sql
✅ Database connection closed
